# Step 1 — Preparing VNTC for TensorFlow

Goal of this notebook: turn the raw VNTC corpus into something `tf.data` can actually read, and
understand every transformation on the way rather than copying a recipe.

The corpus is 84,132 Vietnamese news articles in a folder-per-class layout, which is exactly the
shape `tf.keras.utils.text_dataset_from_directory` expects. Three things stand between the raw
files and a usable dataset, and each is worked through below:

1. the files are **UTF-16LE**, and TensorFlow reads UTF-8
2. the line endings are **CRLF**, this machine is LF
3. whitespace splitting is **syllable-level** for Vietnamese, not word-level

See `../Personal Note.md` for the measurements behind each decision here.

## 1. First, watch it fail

Before fixing anything, point TensorFlow straight at the raw corpus. The failure mode is the part
worth seeing: it does **not** raise.

In [1]:
import tensorflow as tf

raw_ds = tf.keras.utils.text_dataset_from_directory(
    "../data/raw/Train_Full",
    batch_size=2,
    shuffle=False,
)

for texts, labels in raw_ds.take(1):
    print(repr(texts[0].numpy()[:120]))
    break

2026-08-28 11:39:44.714139: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Found 33759 files belonging to 10 classes.


b'\xff\xfe \x00T\x00h\x00\xe0\x00n\x00h\x00 \x00l\x00\xad\x1ep\x00 \x00d\x00\xf1\x1e \x00\xe1\x00n\x00 \x00P\x00O\x00L\x00I\x00C\x00Y\x00 \x00p\x00h\x00\xf2\x00n\x00g\x00 \x00c\x00h\x00\xd1\x1en\x00g\x00 \x00H\x00I\x00V\x00/\x00A\x00I\x00D\x00S\x00 \x00\xdf\x1e \x00V\x00N\x00 \x00(\x00N\x00L\x00\x10\x01)\x00-\x00 \x00Q\x00u\x00'


`Found 33759 files belonging to 10 classes.` — the file discovery works fine, and the count matches
the corpus statistics exactly. But the text comes back as:

```
b'\xff\xfe \x00T\x00h\x00\xe0\x00n\x00h\x00 \x00l\x00\xad\x1ep\x00 ...'
```

That is `" Thành lập dự án POLICY..."` with a null byte after every character, plus the surviving
`\xff\xfe` byte-order mark at the front. It is UTF-16 being read one byte at a time.

**The danger is the silence.** No exception, no warning. `TextVectorization` downstream would build
a vocabulary out of this quite happily, training would run, the loss curve would look plausible, and
the whole thing would be meaningless. This is the class of bug that only shows up if you look.

## 2. Reading it correctly — one call does three jobs

Compare the two ways of getting text out of these files:

In [2]:
p = "../data/raw/Train_Full/The thao/TT_ VNE_ (100).txt"

from_bytes = open(p, "rb").read().decode("utf-16")
text_mode  = open(p, encoding="utf-16").read()

print("open(p,'rb').read().decode('utf-16')  ->", repr(from_bytes[45:62]))
print("open(p, encoding='utf-16').read()     ->", repr(text_mode[45:62]))
print("identical?", from_bytes == text_mode)
print("carriage returns left in text_mode?", "\r" in text_mode)

open(p,'rb').read().decode('utf-16')  -> 'Mở rộng\r\nHạt giốn'
open(p, encoding='utf-16').read()     -> 'Mở rộng\nHạt giống'
identical? False
carriage returns left in text_mode? False


`open(p, encoding="utf-16")` is the better of the two, for reasons that are easy to miss:

- **`"utf-16"`, not `"utf-16-le"`** — the plain name makes Python read the BOM to work out the
  endianness *and consume it*. With `"utf-16-le"` the BOM survives as a stray `\ufeff` character at
  the start of every single document.
- **Text mode defaults to `newline=None`**, i.e. universal-newline mode, which translates `\r\n`
  (and lone `\r`) to `\n` during decoding. The byte-level version preserves CRLF.

Worth being precise about *why* CRLF matters, because the obvious reason is wrong.
`TextVectorization`'s default `split="whitespace"` calls `tf.strings.split(sep=None)`, which treats
any run of whitespace as a separator — `\r` included. So CRLF does **not** produce `"bao\r"`
tokens, and would not have broken training:

```python
tf.strings.split(tf.constant(["Tieu de bai bao\r\nDoan than bai."]))
# [[b'Tieu' b'de' b'bai' b'bao' b'Doan' b'than' b'bai.']]
```

It is normalised anyway because line 0 of every file is the headline: anything that later splits
title from body with `text.split("\n")` gets a trailing `\r` on every headline, invisible when
printed. And because the safety above depends entirely on the default splitter — swap in a custom
one and `\r` is back, since `lower_and_strip_punctuation` does not strip it.

A macOS detail that catches people: Python does **not** fix this on write. `open(p, "w")` translates
`\n` to `os.linesep`, and on POSIX `os.linesep` *is* `\n` — so a string already containing `\r\n`
is written back out verbatim. The fix has to happen on the read side.

## 3. Unicode normalisation — two strings that look identical and are not

Vietnamese diacritics can be encoded two ways, and they do not compare equal:

In [3]:
import unicodedata

nfc = unicodedata.normalize("NFC", "thế")
nfd = unicodedata.normalize("NFD", "thế")

print(f"NFC: {len(nfc)} codepoints {[hex(ord(ch)) for ch in nfc]}")
print(f"NFD: {len(nfd)} codepoints {[hex(ord(ch)) for ch in nfd]}")
print(f"'{nfc}' == '{nfd}' ?  {nfc == nfd}")

NFC: 3 codepoints ['0x74', '0x68', '0x1ebf']
NFD: 5 codepoints ['0x74', '0x68', '0x65', '0x302', '0x301']
'thế' == 'thế' ?  False


Same glyph on screen, different byte sequences, therefore **two separate vocabulary entries** — which
would silently split the evidence for a word in half. VNTC turned out to be 299/300 NFC already
(measured on a 300-file sample), so this costs nothing and removes a whole category of invisible bug.
`260106_TextPreprocessingwithNLP` normalises the same way.

## 4. Word segmentation — why whitespace is not enough for Vietnamese

Vietnamese writes *syllables* separated by spaces, not words. Splitting on whitespace turns
`kinh doanh` (business) into `kinh` + `doanh`, neither of which means much alone, and
`công nghệ thông tin` (information technology) into four unrelated tokens.

`underthesea.word_tokenize(..., format="text")` joins compounds with underscores so they survive as
single tokens:

In [4]:
from underthesea import word_tokenize

sample = "Bộ Công nghiệp vừa công bố mục tiêu xuất khẩu giày dép năm 2005"
print("raw       :", sample)
print("segmented :", word_tokenize(sample, format="text"))

raw       : Bộ Công nghiệp vừa công bố mục tiêu xuất khẩu giày dép năm 2005


segmented : Bộ Công_nghiệp vừa công_bố mục_tiêu xuất_khẩu giày_dép năm 2005


This matters more here than it would in the tutorial's architecture, because the plan is **TF-IDF
features**, where the vocabulary *is* the feature space: `kinh_doanh` as one weighted feature is
strictly more informative than `kinh` and `doanh` as two independent ones.

It also settles the n-gram question. `ngrams=2` was only ever under consideration to glue
`kinh doanh` back together — segmentation does that directly, so running both would pay twice for
the same thing. And this task is keyword detection, not sequence understanding: what separates
`The thao` from `Kinh doanh` is *which words appear*, not their order.

**Cost, measured before committing:** 56 ms/document single-threaded — about 62 minutes for all
84,132 files. Parallelising across processes gets that to roughly 25 minutes. Since `underthesea` is
ordinary Python and cannot run inside a `tf.data` graph pipeline without `tf.py_function` (losing
the graph's parallelism), and since re-running it every epoch would be absurd, it happens **once,
here, written to disk**.

## 5. The conversion pass

Everything above, applied to all 84,132 files:

| | |
|---|---|
| read | `open(p, encoding="utf-16")` — BOM consumed, CRLF → LF |
| normalise | `unicodedata.normalize("NFC", ...)` |
| segment | `underthesea.word_tokenize(..., format="text")` |
| write | UTF-8 into `../data/processed/`, folder-per-class preserved |

Two implementation choices worth stating. It is **resumable** — a file whose output already exists
is skipped, so an interrupted run costs only what it had not finished. And the folder-per-class
layout is preserved exactly, because that is where `text_dataset_from_directory` gets the labels
from; flattening it would throw the labels away.

### The `mp_context="fork"` is not decoration

Running a `ProcessPoolExecutor` **from a notebook** on macOS needs one non-obvious fix. Python 3.12
defaults to the **spawn** start method on macOS: each worker is a fresh interpreter that re-imports
the target function by module path. A function defined in a notebook cell lives in `__main__`, which
a spawned worker cannot import — so the naive version fails with a pickling error.

`multiprocessing.get_context("fork")` copies the parent's memory instead of re-importing, so
`convert_one` comes along with it. Measured on this machine: 8 forked workers do the job in about
**28 minutes** versus 62 sequential.

Threads are not an alternative here — `ThreadPoolExecutor` was measured at only **1.36×**, because
`underthesea` holds the GIL for most of its work. Segmentation is CPU-bound Python, and CPU-bound
Python needs processes.

In [5]:
import os, time, unicodedata
import multiprocessing as mp
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

RAW       = Path("../data/raw")
PROCESSED = Path("../data/processed")
SPLITS    = ["Train_Full", "Test_Full"]


def convert_one(job):
    src, dst = job
    if dst.exists():                       # resumable: already done
        return 0
    text = open(src, encoding="utf-16").read()      # BOM + CRLF handled here
    text = unicodedata.normalize("NFC", text)
    from underthesea import word_tokenize            # imported per worker process
    segmented = word_tokenize(text, format="text")
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text(segmented, encoding="utf-8")
    return 1


jobs = []
for split in SPLITS:
    for class_dir in sorted((RAW / split).iterdir()):
        if not class_dir.is_dir():
            continue
        for src in class_dir.glob("*.txt"):
            jobs.append((src, PROCESSED / split / class_dir.name / src.name))

todo = [j for j in jobs if not j[1].exists()]
print(f"{len(jobs)} files total, {len(todo)} still to convert")

t0 = time.time()
done = 0
# fork, not the macOS default spawn: convert_one is defined in this notebook's __main__
# and a spawned worker could not import it
with ProcessPoolExecutor(max_workers=8, mp_context=mp.get_context("fork")) as pool:
    for result in pool.map(convert_one, todo, chunksize=64):
        done += result
        if done and done % 5000 == 0:
            rate = done / (time.time() - t0)
            print(f"  {done:6}/{len(todo)}  {rate:5.0f} files/s  "
                  f"ETA {(len(todo)-done)/rate/60:4.1f} min")

print(f"converted {done} files in {(time.time()-t0)/60:.1f} min")

84132 files total, 84132 still to convert


    5000/84132     80 files/s  ETA 16.5 min


   10000/84132     72 files/s  ETA 17.1 min


   15000/84132     76 files/s  ETA 15.2 min


   20000/84132     80 files/s  ETA 13.4 min


...

   65000/84132     69 files/s  ETA  4.6 min


   70000/84132     67 files/s  ETA  3.5 min


   75000/84132     65 files/s  ETA  2.3 min


   80000/84132     63 files/s  ETA  1.1 min


converted 84132 files in 21.7 min


## 6. Verify before trusting it

Three things have to be true, and each failed silently earlier in this notebook if it were wrong:
the file count matches, the diacritics survived, and the compounds are actually joined.

In [6]:
for split in SPLITS:
    n_raw  = sum(1 for _ in (RAW / split).rglob("*.txt"))
    n_out  = sum(1 for _ in (PROCESSED / split).rglob("*.txt"))
    print(f"{split:11} raw={n_raw:6}  processed={n_out:6}  {'OK' if n_raw == n_out else 'MISMATCH'}")

sample = next((PROCESSED / "Train_Full" / "The thao").glob("*.txt"))
text = sample.read_text(encoding="utf-8")
print(f"\n{sample.name}")
print("  text     :", text[:150])
print("  has '_'  :", "_" in text)
print("  has '\\r' :", "\r" in text)
print("  is NFC   :", text == unicodedata.normalize("NFC", text))

Train_Full  raw= 33759  processed= 33759  OK


Test_Full   raw= 50373  processed= 50373  OK

TT_TN_ (4608).txt
  text     : Các đội đầu_bảng đã nỗ_lực gia_tăng cách_biệt , trong khi nhóm giữa tiếp_tục kèn_cựa nhau bằng những cuộc chia_điểm . Bất_ngờ lớn nhất là trận thắng đ
  has '_'  : True
  has '\r' : False
  is NFC   : True


## 7. Load the processed corpus with `tf.data`

Now the part this project actually exists to learn. `text_dataset_from_directory` returns a
`tf.data.Dataset`, which is **not** a list — it is a lazy pipeline that yields batches on demand,
so 500 MB of text never has to sit in memory at once.

Three things to understand about what follows:

- a single element of the dataset is a **batch** `(texts, labels)`, not one document
- `validation_split` carves the validation set off the **training** data — the official test set is
  not touched until Step 4, and adapting or tuning on it would be leakage
- the same `seed` in both calls is what guarantees the two subsets do not overlap

In [7]:
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.text_dataset_from_directory(
    PROCESSED / "Train_Full",
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="training",
    seed=SEED,
)

val_ds = tf.keras.utils.text_dataset_from_directory(
    PROCESSED / "Train_Full",
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
)

class_names = train_ds.class_names
print("\nclasses:", class_names)

Found 33759 files belonging to 10 classes.


Using 27008 files for training.


Found 33759 files belonging to 10 classes.


Using 6751 files for validation.



classes: ['Chinh tri Xa hoi', 'Doi song', 'Khoa hoc', 'Kinh doanh', 'Phap luat', 'Suc khoe', 'The gioi', 'The thao', 'Van hoa', 'Vi tinh']


Labels are **integers 0-9**, assigned by sorting the folder names alphabetically. That integer
encoding is why Step 3 uses `SparseCategoricalCrossentropy` rather than the one-hot
`CategoricalCrossentropy` — same maths, but it takes the labels in the format we already have.

Check that the labels line up with the folders they came from:

In [8]:
for texts, labels in train_ds.take(1):
    for i in range(3):
        print(f"[{labels[i].numpy()}] {class_names[labels[i]]:18} {texts[i].numpy().decode()[:90]}")
    break

[6] The gioi           Thị_trưởng Tokyo chỉ_trích Trung_Quốc Hôm_qua thị_trưởng Tokyo bình_luận rằng chính_quyền 
[2] Khoa hoc           Băng Greenland có nguy_cơ biến mất Khối_băng khổng_lồ Greenland có_thể tan biến trong 1.00
[2] Khoa hoc           Căn nhà thông_thái dè_sẻn năng_lượng Tại sân_bay Dallas / Fort_Worth , các bóng_đèn được k


## What is left for Step 2

The processed corpus is on disk and loads cleanly. Step 2 turns it into TF-IDF features with a
`TextVectorization` layer — and the first thing to check there is that the default
`lower_and_strip_punctuation` standardiser **strips underscores**, which would silently undo every
compound this notebook just spent 25 minutes building.